# 08_2 — 全行列の単独LRAと両端bulkKS順の累積LRA

Llama・GemmaのQ/K/V/O/gate/up/downを**各行列単独・全評価文章**で評価します。
実験3は保存済みの `condition=DE_1`, `method=matched_upper`, `two_sided_bulkKS` の**降順**。
rankは08と同じ：単独比較ではDE前MP rankを共通使用。累積比較ではOffDEはDE前MP、DEは分散1のMP rank。
下側成分をLRAで残す変更はありません。変更は累積対象の順序のみです。

**本番未実行。** `RUN_EXPERIMENTS=True` にして実行してください。08は変更せず、新しい保存先を使用します。
既存08の同条件・全量評価の単独結果とbaselineを検証してコピーし、未完了のみ実行します。
全計画916条件（単独868＋累積48）。既存88条件を再利用できれば追加828条件です。
PPL・baseline比・DE改善率で集計します。損失はPPL再計算用に内部保存します。


## 1. 環境
VS CodeからColabに接続する場合も、HF認証は環境変数または下の入力欄を使用します。Colab secretsを呼びません。最初にRMT_utilsが必要です。ローカルでは既存checkout、Colabでは未存在の場合だけGitHubから取得します。インストール後にカーネル再起動が求められた場合は再起動してください。

In [4]:
from pathlib import Path
import os, sys, subprocess, getpass
INSTALL_DEPENDENCIES = False
CLONE_IF_MISSING = True
REPO = Path('/content/RMT_utils') if Path('/content').exists() else Path.home() / 'GitHub/RMT_utils'
if not (REPO / 'funcs1.py').exists():
    if not CLONE_IF_MISSING:
        raise FileNotFoundError(f'RMT_utils not found: {REPO}')
    subprocess.run(['git', 'clone', 'https://github.com/ryy1210/RMT_utils', str(REPO)], check=True)
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements-notebooks.txt')], check=True)
sys.path.insert(0, str(REPO))
# 必要なときだけ入力。キーをセル出力・設定ファイルに保存しない。
if not os.environ.get('HF_TOKEN'):
    token = getpass.getpass('HF_TOKEN (cached loginなら空欄): ')
    if token: os.environ['HF_TOKEN'] = token
print('Repository:', REPO)

Repository: /content/RMT_utils


## 2. 設定
実験1の予算は「モデル全体のパラメータに対する割合」。言語モデルのみのパラメータ数を分母とし、Gemmaのvision towerは含めません。共通予算が実現不能なら停止するため、勝手に小さく丸めません。
実験2はpreDE MP rank固定で、各モデルの最初・中央・最後のblockを使います。全行列探索は既定OFFです。
実験3の既定は0721と同じ `KS_postDE_1_saved` 降順・postDE BEMA rank。分散1版は `POST_RANK_SOURCE='fixed1'` で別TAGにします。今回の固定rank条件はこの設定に影響されません。

rank=0は元のMP規則の一部として許可し計画表に明示します。因子化が密行列より大きい場合もrankを改変せず、密行列で保持した場合の削減量0を記録します。

In [5]:
import json, hashlib, math, gc, platform, re, time, importlib.metadata
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from funcs1 import apply_lra_1

MOUNT_DRIVE = True  # Colabで未マウントならTrue。VS CodeのDrive接続済みならFalse
if MOUNT_DRIVE and Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
LOCAL_DRIVE = Path.home()/'Library/CloudStorage/GoogleDrive-ryoya.zushi1210@gmail.com/マイドライブ/TUS/hashiguchi/data'
DATA_ROOT = Path('/content/drive/MyDrive/TUS/hashiguchi/data') if Path('/content').exists() else LOCAL_DRIVE
SOURCE_TAG = 'bulk-ks-20260925-v1-full'
EXPERIMENT_TAG = 'ibis2026-de-controlled-v2-bulk-order'
LEGACY_OUTPUT = DATA_ROOT/'ibis2026_de_lra'/'ibis2026-de-controlled-v1'
BULK_PATH = DATA_ROOT/'bulk_ks_two_sided'/'two-sided-bulk-ks-v1'/'two_sided_bulk_ks.csv'
OUTPUT = DATA_ROOT/'ibis2026_de_lra'/EXPERIMENT_TAG
MODELS = {'llama':'meta-llama/Llama-3.2-3B', 'gemma':'google/gemma-3-4b-pt'}
MODEL_REVISIONS = {k:'main' for k in MODELS}  # 初回計画時にcommitへ解決して固定
MODULES = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
MODEL_BUDGETS = [0.0025, 0.005, 0.01]
RUN_EXPERIMENTS = True  # Trueなら実験を実行、Falseなら既存結果を再利用
REUSE_08 = True  # 条件一致を検証した単独全量評価のみ再利用
POST_RANK_SOURCE = 'fixed1'  # 'bema' reproduces 0721; 'fixed1' uses n_signal_DE_1
ORDER_COLUMN = 'two_sided_bulkKS_DE_1'
ORDER_METHOD = 'matched_upper'
MAX_CUMULATIVE = 100
CUMULATIVE_STEPS = [1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
SEQ_LEN = 1024
BATCH_SIZE = 1
SCREEN_BLOCKS = 16
BOOTSTRAP_REPEATS = 2000
SEED = 42
PUBLISH_WANDB = True
WANDB_ENTITY = 'ryoya-zushi1210-tokyo-university-of-science'
WANDB_PROJECT = 'IBIS2026-DE-LRA'
AUTO_DISCONNECT = False
# PPLは両モデルで直接比較せず、同じtokenizerのbaselineとの差を比較する。
plt.rcParams.update({'figure.figsize':(9,4), 'font.size':11, 'axes.spines.top':False, 'axes.spines.right':False})

def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def write_json(path, data):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)
def write_csv(path, frame):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp=path.with_suffix('.tmp.csv'); frame.to_csv(temp,index=False); temp.replace(path)
def safe_exp(x): return math.exp(x) if x < 709 else float('inf')
# 変更: 08と同一checkpointを使う。保存済みmanifestから固定revisionを取得。
LEGACY_MANIFEST=json.loads((LEGACY_OUTPUT/'manifest.json').read_text())
MODEL_REVISIONS=LEGACY_MANIFEST['settings']['revisions'].copy()
BULK=pd.read_csv(BULK_PATH)
BULK=BULK[(BULK.condition=='DE_1') & (BULK.method==ORDER_METHOD)].copy()
if BULK.duplicated(['model','name']).any(): raise ValueError('Duplicate bulkKS rows')
if not (BULK.status.eq('ok') & np.isfinite(BULK.two_sided_bulkKS) & BULK.two_sided_bulkKS.between(0,1) & np.isclose(BULK.sigma2,1)).all():
    raise ValueError('Invalid DE_1 bulkKS source')
print('Saved two-sided bulkKS:', BULK.groupby('model').size().to_dict())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved two-sided bulkKS: {'gemma': 238, 'llama': 196}


## 3. 入力確認とモデルサイズ
モデルの構成だけを読み、meta device上で形状とパラメータ数を取得します。このセルではモデル重みをダウンロードしません。保存rankと再計算rankの一致・全7モジュールの網羅・行列形状を検証します。

In [6]:
def load_metadata(key):
    path=DATA_ROOT/'bulk_ks'/key/SOURCE_TAG/'bulk_ks.csv'
    df=pd.read_csv(path)
    bulk=BULK[BULK.model==key]
    if set(bulk.name)!=set(df.name): raise ValueError(f'{key}: bulkKS coverage mismatch')
    if not np.array_equal(bulk.set_index('name').loc[df.name,['p','n']].to_numpy(),df[['p','n']].to_numpy()):
        raise ValueError('bulkKS shape mismatch')
    df=df.merge(bulk[['name','two_sided_bulkKS']].rename(columns={'two_sided_bulkKS':ORDER_COLUMN}),on='name',validate='one_to_one')
    required=['name','p','n','module_type','block_index','s_hat_saved_preDE','s_hat_saved_postDE','n_signal_DE_1',ORDER_COLUMN]
    if not set(required)<=set(df): raise ValueError(f'Missing columns: {set(required)-set(df)}')
    if df.name.duplicated().any(): raise ValueError('Duplicate matrix names')
    if set(df.module_type)!=set(MODULES): raise ValueError('Module coverage mismatch')
    for col in ['rank_matches_saved_preDE','rank_matches_saved_postDE']:
        if not df[col].astype(str).str.lower().eq('true').all(): raise ValueError(f'{col}: mismatch')
    for col in ['s_hat_saved_preDE','s_hat_saved_postDE','n_signal_DE_1']:
        x=pd.to_numeric(df[col],errors='raise')
        if not (np.isfinite(x)&(x==np.floor(x))&(x>=0)&(x<=df.p)).all(): raise ValueError(f'Invalid rank: {col}')
    if not np.isfinite(df[ORDER_COLUMN]).all(): raise ValueError('Invalid order metric')
    config=AutoConfig.from_pretrained(MODELS[key],revision=MODEL_REVISIONS[key],token=os.environ.get('HF_TOKEN'))
    # 修正: 保存済みcommit SHAを正とする。内部属性が空でも固定指定は失われていない。
    revision=MODEL_REVISIONS[key]
    if not isinstance(revision,str) or not re.fullmatch(r'[0-9a-f]{40}',revision):
        raise ValueError('Saved model revision is not a full commit SHA')
    resolved=getattr(config,'_commit_hash',None)
    if resolved and resolved!=revision:
        raise ValueError(f'Model revision mismatch: {resolved} != {revision}')
    # Gemma3 multimodal configから言語モデルだけを作る。
    text_config=getattr(config,'text_config',config)
    with torch.device('meta'):
        shell=AutoModelForCausalLM.from_config(text_config)
        shell.tie_weights()
    total=sum(p.numel() for p in shell.parameters())
    modules=dict(shell.named_modules())
    for row in df.itertuples():
        matches=[name for name,mod in modules.items() if isinstance(mod,torch.nn.Linear) and (name==row.name or name.endswith('.'+row.name))]
        if len(matches)!=1: raise ValueError(f'Cannot resolve {row.name}: {matches}')
        m,n=modules[matches[0]].weight.shape
        if sorted([m,n])!=[int(row.p),int(row.n)]: raise ValueError(f'Shape mismatch {row.name}')
    del shell
    return df.sort_values(['block_index','module_type']).reset_index(drop=True), int(total), revision, sha(path)

METRICS={}; TOTAL={}; REVISIONS={}; SOURCE_HASH={}
for key in MODELS:
    METRICS[key],TOTAL[key],REVISIONS[key],SOURCE_HASH[key]=load_metadata(key)
    print(key, 'matrices:',len(METRICS[key]), 'language parameters:',TOTAL[key], 'revision:',REVISIONS[key])

ValueError: Could not pin model revision

## 4. 全行列単独比較と新しい累積順序の計画
同じTAGで条件を変えた場合は停止します。再実行は完了済み条件をスキップします。

In [ ]:
def cost(row,r): return min(int(row.p)*int(row.n),int(r)*(int(row.p)+int(row.n)))
def rank_map(df,col): return {r['name']:int(r[col]) for _,r in df.iterrows()}
def budget_ranks(df,total,budget):
    original=int((df.p*df.n).sum()); target=float(budget)*total
    if target>=original: raise ValueError(f'Budget {budget:.2%} exceeds module size {original/total:.2%}')
    fraction=target/original
    # ceil: 指定予算を超えて削減しない。r<min(shape)を維持。
    return {r['name']:min(int(r.p),max(1,math.ceil((1-fraction)*r.p*r.n/(r.p+r.n)))) for _,r in df.iterrows()}
def make_plan():
    jobs=[]
    def add(key,exp,variant,scope,selected,ranks,use_de,step=0):
        original=int((selected.p*selected.n).sum())
        saved=sum(int(r.p*r.n)-cost(r,ranks[r['name']]) for _,r in selected.iterrows())
        identity=f'{key}__{exp}__{variant}__{scope}__step{step}__{int(use_de)}'
        jobs.append(dict(run_id=identity,model=key,experiment=exp,variant=variant,scope=scope,step=step,use_de=use_de,
                         names=list(selected.name),ranks=ranks,selected_count=len(selected),params_saved=saved,
                         reduction_model=saved/TOTAL[key],reduction_target=saved/original,
                         zero_ranks=sum(r==0 for r in ranks.values())))
    for key,df in METRICS.items():
        # 変更: 434行列を一つずつ、08の単独実験と同じ全量評価で比較。
        for i in range(len(df)):
            one=df.iloc[[i]]
            for de in [False,True]: add(key,'E2','all_single',one.iloc[0]['name'],one,rank_map(one,'s_hat_saved_preDE'),de)
        ordered=df.sort_values([ORDER_COLUMN,'name'],ascending=[False,True],kind='stable').head(MAX_CUMULATIVE)
        for step in sorted(set([s for s in CUMULATIVE_STEPS if s<=len(ordered)]+[len(ordered)])):
            selected=ordered.head(step)
            for de in [False,True]:
                col=('s_hat_saved_postDE' if POST_RANK_SOURCE=='bema' else 'n_signal_DE_1') if de else 's_hat_saved_preDE'
                add(key,'E3',f'bulk_two_sided_{POST_RANK_SOURCE}','common_bulkKS_order',selected,rank_map(selected,col),de,step)
    return jobs

if POST_RANK_SOURCE not in ['bema','fixed1']: raise ValueError('Invalid POST_RANK_SOURCE')
JOBS=make_plan()
plan=pd.DataFrame([{k:v for k,v in j.items() if k not in ['names','ranks']} for j in JOBS])
display(plan.groupby(['model','experiment','variant']).size().rename('evaluations').to_frame())
display(plan[plan.experiment=='E3'][['model','variant','scope','use_de','reduction_model','reduction_target','zero_ranks']])
print('Planned evaluations:',len(JOBS),' + one baseline per model. No model weights loaded yet.')
SETTINGS=dict(models=MODELS,revisions=REVISIONS,source=SOURCE_HASH,total=TOTAL,jobs=JOBS,
              seq_len=SEQ_LEN,batch_size=BATCH_SIZE,screen_blocks=SCREEN_BLOCKS,seed=SEED,
              dataset='Salesforce/wikitext:wikitext-2-raw-v1:test',dtype='float32 SVD / bfloat16 inference',
              rank_source=POST_RANK_SOURCE,bulk_source_sha256=sha(BULK_PATH),order_column=ORDER_COLUMN,order_method=ORDER_METHOD,code_hashes={f:sha(REPO/f) for f in ['funcs1.py','evaluater.py']})
CONFIG_HASH=hashlib.sha256(json.dumps(SETTINGS,sort_keys=True).encode()).hexdigest()
OUTPUT.mkdir(parents=True,exist_ok=True)
manifest=OUTPUT/'manifest.json'
if manifest.exists() and json.loads(manifest.read_text())['config_hash']!=CONFIG_HASH:
    raise RuntimeError('Existing TAG has different settings. Choose a new EXPERIMENT_TAG.')
write_json(manifest,dict(config_hash=CONFIG_HASH,settings=SETTINGS))
write_csv(OUTPUT/'plan.csv',plan)
write_json(OUTPUT/'rank_plan.json',JOBS)
# rankと実際の並び順を保存。KSは順序だけに使用し、rankを決め直さない。
for key,df in METRICS.items():
    write_csv(OUTPUT/f'{key}_bulk_order.csv',df.sort_values([ORDER_COLUMN,'name'],ascending=[False,True]))


evaluations
model experiment variant                           
gemma E2         all_single                     476
      E3         bulk_two_sided_fixed1           24
llama E2         all_single                     392
      E3         bulk_two_sided_fixed1           24

,model,variant,scope,use_de,reduction_model,reduction_target,zero_ranks
392,llama,bulk_two_sided_fixed1,common_bulkKS_order,False,0.001819,0.619141,0
393,llama,bulk_two_sided_fixed1,common_bulkKS_order,True,0.001580,0.537760,0
394,llama,bulk_two_sided_fixed1,common_bulkKS_order,False,0.007391,0.686257,0
395,llama,bulk_two_sided_fixed1,common_bulkKS_order,True,0.007166,0.665305,0
396,llama,bulk_two_sided_fixed1,common_bulkKS_order,False,0.015638,0.725941,0
397,llama,bulk_two_sided_fixed1,common_bulkKS_order,True,0.015076,0.699870,0
398,llama,bulk_two_sided_fixed1,common_bulkKS_order,False,0.038425,0.754695,0
399,llama,bulk_two_sided_fixed1,common_bulkKS_order,True,0.037363,0.733824,0
400,llama,bulk_two_sided_fixed1,common_bulkKS_order,False,0.053624,0.760652,0
401,llama,bulk_two_sided_fixed1,common_bulkKS_order,True,0.052499,0.744683,0


Planned evaluations: 916  + one baseline per model. No model weights loaded yet.


## 5. 固定評価データとモデル読み込み
WikiText-2 testを文書順に連結し、重複しない1024 tokenブロックで評価します。各ブロック先頭tokenと末尾の不完全ブロックは評価から除外し、有効token数を保存します。過去のPPL数値をそのままbaselineにせず、この手順で再評価します。ブロック単位の対応bootstrapは、この評価コーパスに対する不確実性であり、モデル母集団への一般化ではありません。
同じ設定の再実行では保存済みtoken列・commitを再利用します。

In [ ]:
GEMMA_LOADER_VERSION = 'conditional_verified_v2'

def result_root(key):
    # 修正: Gemmaの旧baseline・結果・近似キャッシュを一切再利用しない。
    # Llamaの保存場所とconfig hashは維持する。
    return OUTPUT/'gemma_loader_v2' if key=='gemma' else OUTPUT

def run_directory(job):
    return result_root(job['model'])/'runs'/job['run_id']

def check_loading_info(info):
    if not isinstance(info,dict): raise RuntimeError('Loading report missing')
    errors={k:info.get(k) for k in ['missing_keys','unexpected_keys','mismatched_keys','error_msgs'] if info.get(k)}
    if errors: raise RuntimeError(f'Pretrained checkpoint did not load completely: {errors}')

def extract_gemma_text(full):
    from transformers import Gemma3ForCausalLM
    # 正式クラスでロードした実Tensorを共有。meta上の空モデルへ全モジュールを置換する。
    decoder=full.model.language_model
    with torch.device('meta'):
        text=Gemma3ForCausalLM(full.config.text_config)
    text.model=decoder
    text.lm_head=full.lm_head
    text.generation_config=full.generation_config
    if any(p.is_meta for p in text.parameters()) or any(b.is_meta for b in text.buffers()):
        raise RuntimeError('Unloaded meta tensors remain')
    return text

def model_load(key):
    kwargs=dict(revision=REVISIONS[key],token=os.environ.get('HF_TOKEN'),torch_dtype=torch.bfloat16,
                low_cpu_mem_usage=True,output_loading_info=True)
    if key=='gemma':
        from transformers import Gemma3ForConditionalGeneration
        # 修正: checkpointのarchitectureと一致するクラスを使う。text_configだけを渡さない。
        full,info=Gemma3ForConditionalGeneration.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
        model=extract_gemma_text(full)
        del full; gc.collect()
    else:
        model,info=AutoModelForCausalLM.from_pretrained(MODELS[key],**kwargs)
        check_loading_info(info)
    actual=sum(p.numel() for p in model.parameters())
    if actual!=TOTAL[key]: raise ValueError(f'Parameter count mismatch {actual} != {TOTAL[key]}')
    if any(p.is_meta for p in model.parameters()): raise RuntimeError('Unloaded model parameters')
    root=result_root(key); root.mkdir(parents=True,exist_ok=True)
    write_json(root/'loading_report.json',dict(model=MODELS[key],revision=REVISIONS[key],
        loader_version=GEMMA_LOADER_VERSION if key=='gemma' else 'auto_verified_v2',
        transformers=importlib.metadata.version('transformers'),parameter_count=actual,
        missing_keys=[],unexpected_keys=[],mismatched_keys=[],config_hash=CONFIG_HASH))
    model.eval(); model.config.use_cache=False
    return model.to('cuda')

def tokens_load(key):
    path=OUTPUT/f'{key}_tokens.npy'
    if path.exists():
        receipt=json.loads((OUTPUT/f'{key}_tokens.json').read_text())
        if receipt['sha256']!=sha(path): raise ValueError('Cached tokens changed')
        return np.load(path)
    tok=AutoTokenizer.from_pretrained(MODELS[key],revision=REVISIONS[key],token=os.environ.get('HF_TOKEN'))
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split='test')
    ids=tok('\n\n'.join(ds['text']),add_special_tokens=False)['input_ids']
    count=len(ids)//SEQ_LEN
    if count<SCREEN_BLOCKS: raise ValueError('Too few evaluation blocks')
    values=np.asarray(ids[:count*SEQ_LEN],dtype=np.int64).reshape(count,SEQ_LEN)
    np.save(path,values)
    write_json(OUTPUT/f'{key}_tokens.json',dict(dataset_fingerprint=ds._fingerprint,revision=REVISIONS[key],
               seq_len=SEQ_LEN,blocks=count,valid_tokens=count*(SEQ_LEN-1),discarded_tokens=len(ids)-count*SEQ_LEN,sha256=sha(path)))
    return values
@torch.inference_mode()
def evaluate(model,tokens):
    losses=[]; counts=[]
    for start in range(0,len(tokens),BATCH_SIZE):
        ids=torch.as_tensor(tokens[start:start+BATCH_SIZE],device=next(model.parameters()).device)
        logits=model(input_ids=ids,use_cache=False).logits
        sums=torch.zeros(ids.shape[0],device=ids.device,dtype=torch.float64)
        for pos in range(0,ids.shape[1]-1,64):
            end=min(pos+64,ids.shape[1]-1)
            loss=F.cross_entropy(logits[:,pos:end].float().transpose(1,2),ids[:,pos+1:end+1],reduction='none')
            if not torch.isfinite(loss).all(): raise FloatingPointError('Nonfinite NLL')
            sums+=loss.double().sum(1)
        losses.extend(sums.cpu().tolist()); counts.extend([ids.shape[1]-1]*ids.shape[0]); del logits
    return np.asarray(losses),np.asarray(counts)
def resolve(model,name):
    found=[m for k,m in model.named_modules() if isinstance(m,torch.nn.Linear) and (k==name or k.endswith('.'+name))]
    if len(found)!=1: raise ValueError(f'Ambiguous module {name}')
    return found[0]

def save_eval(directory,loss,counts,info):
    directory.mkdir(parents=True,exist_ok=True)
    np.savez_compressed(directory/'block_losses.npz',loss_sum=loss,token_count=counts)
    nll=float(loss.sum()/counts.sum())
    # InfinityをJSONに書かず、NLLを主指標として保持する。
    ppl=safe_exp(nll)
    write_json(directory/'summary.json',{**info, 'loader_version': GEMMA_LOADER_VERSION if info.get('model')=='gemma' else 'legacy_llama_compatible','nll':nll,'ppl':ppl if np.isfinite(ppl) else None,'tokens':int(counts.sum()),'status':'ok'})

def completed(directory):
    if not ((directory/'summary.json').exists() and (directory/'block_losses.npz').exists()): return False
    info=json.loads((directory/'summary.json').read_text())
    if info.get('model')=='gemma' and info.get('loader_version')!=GEMMA_LOADER_VERSION: return False
    if info.get('config_hash')!=CONFIG_HASH: raise ValueError('Result config mismatch')
    return info.get('status')=='ok'

def import_08_results():
    """変更: 08の出力を変更せず、互換性のあるbaseline/単独結果のみコピーする。"""
    import shutil
    if not REUSE_08: return
    old=LEGACY_MANIFEST['settings']; old_hash=LEGACY_MANIFEST['config_hash']
    for field in ['models','revisions','source','total','seq_len','batch_size','dataset','dtype','code_hashes']:
        if old[field]!=SETTINGS[field]: raise ValueError(f'08 reuse incompatible: {field}')
    oldjobs=old['jobs']
    def legacy_root(key): return LEGACY_OUTPUT/'gemma_loader_v2' if key=='gemma' else LEGACY_OUTPUT
    def validated(path,key):
        info=json.loads((path/'summary.json').read_text())
        if info.get('status')!='ok' or info.get('config_hash')!=old_hash: raise ValueError(f'Invalid old result {path}')
        if key=='gemma' and info.get('loader_version')!=GEMMA_LOADER_VERSION: raise ValueError('Old Gemma loader')
        if info['token_sha256']!=sha(LEGACY_OUTPUT/f'{key}_tokens.npy'): raise ValueError('Old token hash mismatch')
        with np.load(path/'block_losses.npz') as z: loss=z['loss_sum']; count=z['token_count']
        tok=np.load(LEGACY_OUTPUT/f'{key}_tokens.npy',mmap_mode='r')
        if len(loss)!=len(tok) or not np.all(count==SEQ_LEN-1) or not np.isfinite(loss).all(): raise ValueError('Not full evaluation')
        if not np.isclose(loss.sum()/count.sum(),info['nll'],rtol=1e-10): raise ValueError('Saved loss mismatch')
        return info
    def copy_result(source,dest,info):
        dest.mkdir(parents=True,exist_ok=True)
        for name in ['block_losses.npz','matrices.csv']:
            if (source/name).exists(): shutil.copy2(source/name,dest/name)
        info={**info,'config_hash':CONFIG_HASH,'reused_from':str(source),'original_config_hash':old_hash}
        write_json(dest/'summary.json',info)  # 完了マーカーは最後に書く
    count=0
    for key in MODELS:
        # 08の固定トークン列をそのまま使い、評価文章の変更を防ぐ。
        for suffix in ['npy','json']:
            source=LEGACY_OUTPUT/f'{key}_tokens.{suffix}'; dest=OUTPUT/source.name
            if dest.exists():
                if sha(source)!=sha(dest): raise ValueError('Existing token file differs')
            else: shutil.copy2(source,dest)
        basepath=legacy_root(key)/'baselines'/key
        base=validated(basepath,key)
        dest=result_root(key)/'baselines'/key
        if not completed(dest): copy_result(basepath,dest,base)
        for job in [j for j in JOBS if j['model']==key and j['experiment']=='E2']:
            dest=run_directory(job)
            if completed(dest): continue
            matches=[j for j in oldjobs if j['model']==key and j['experiment']=='E2' and j['variant']!='screen'
                     and j['names']==job['names'] and j['ranks']==job['ranks'] and j['use_de']==job['use_de']]
            if not matches: continue
            oldjob=matches[0]; source=legacy_root(key)/'runs'/oldjob['run_id']
            if not (source/'summary.json').exists(): continue
            info=validated(source,key)
            frame=pd.read_csv(source/'matrices.csv')
            if dict(zip(frame.name,frame['rank']))!=job['ranks']: raise ValueError('Old ranks mismatch')
            if info['params_saved']!=job['params_saved'] or not np.isclose(info['baseline_nll'],base['nll']): raise ValueError('Old comparison mismatch')
            info.update({k:v for k,v in job.items() if k not in ['names','ranks']})
            copy_result(source,dest,info); count+=1
    print('08から再利用した単独評価:',count)

import_08_results()
print('未完了条件:',sum(not completed(run_directory(j)) for j in JOBS),'/',len(JOBS))


08から再利用した単独評価: 0
未完了条件: 55 / 916


## 6. 本番実行
モデルは一度に一つだけGPUへロードします。各試行で変更する重みだけCPUへ退避し、`finally`で復元します。実験3も各checkpointを元モデルから再構成するので、途中再開で二重LRAになりません。**計算節約のため、同じ行列・DE条件・rankの近似重みをディスクにキャッシュします。** 大きなキャッシュ領域が必要です（不要になったら手動削除）。実験1/2/3の結果を混同しないよう別runとして保存します。失敗時はその場で停止し、完了済みrunは再実行しません。

In [ ]:
def approximation(key,name,original,rank,use_de):
    signature=hashlib.sha256(f'{CONFIG_HASH}:{key}:{name}:{rank}:{use_de}'.encode()).hexdigest()
    directory=result_root(key)/'approximation_cache'; directory.mkdir(exist_ok=True)
    path=directory/f'{signature}.pt'
    if path.exists(): return torch.load(path,map_location='cpu',weights_only=True)
    # 変更: 08で同じ重み・rank・DE条件の近似があればSVDを再実行しない。
    # import_08_resultsでrevision・ソース・実装hash等の一致を検証済み。
    if REUSE_08:
        old_hash=LEGACY_MANIFEST['config_hash']
        old_signature=hashlib.sha256(f'{old_hash}:{key}:{name}:{rank}:{use_de}'.encode()).hexdigest()
        old_root=LEGACY_OUTPUT/'gemma_loader_v2' if key=='gemma' else LEGACY_OUTPUT
        old_path=old_root/'approximation_cache'/f'{old_signature}.pt'
        if old_path.exists():
            return torch.load(old_path,map_location='cpu',weights_only=True)
    # 既存DE関数を使用。float32で計算し、推論dtypeへの丸めはcopy時のみ。
    value=apply_lra_1(original.float(),rank,DE=use_de,fast_SVD=False).cpu()
    temp=path.with_suffix('.tmp'); torch.save(value,temp); temp.replace(path)
    return value

def execute_job(model,key,job,tokens,baseline):
    directory=run_directory(job)
    if completed(directory): return
    backups={}; records=[]; start=time.time()
    try:
        for name in job['names']:
            module=resolve(model,name); original=module.weight.detach().cpu().clone()
            backups[name]=original
            rank=job['ranks'][name]
            approx=approximation(key,name,original,rank,job['use_de'])
            if approx.shape!=original.shape or not torch.isfinite(approx).all(): raise ValueError('Invalid approximation')
            relative=float(torch.linalg.vector_norm(approx-original.float())/torch.linalg.vector_norm(original.float()))
            with torch.no_grad(): module.weight.copy_(approx.to(device=module.weight.device,dtype=module.weight.dtype))
            record=dict(name=name,rank=rank,rows=original.shape[0],columns=original.shape[1],relative_frobenius_error=relative)
            if key in METRICS:
                source=METRICS[key].set_index('name').loc[name]
                for col in ['module_type','block_index','KS_preDE','KS_postDE','KS_postDE_1','bulkKS_preDE','bulkKS_postDE','bulkKS_DE_1']:
                    if col in source: record[col]=source[col]
            records.append(record)
        short=False  # 変更: 全行列を08の既存単独評価と同じ全文章で評価
        evaluated=tokens[:SCREEN_BLOCKS] if short else tokens
        losses,counts=evaluate(model,evaluated)
        base_loss,base_count=baseline
        base_nll=float(base_loss[:len(losses)].sum()/base_count[:len(losses)].sum())
        write_csv(directory/'matrices.csv',pd.DataFrame(records))
        save_eval(directory,losses,counts,{k:v for k,v in job.items() if k not in ['names','ranks']} | dict(
            config_hash=CONFIG_HASH,token_sha256=sha(OUTPUT/f'{key}_tokens.npy'),baseline_nll=base_nll,
            elapsed_seconds=time.time()-start,evaluation='screen' if short else 'full'))
    finally:
        for name,original in backups.items():
            module=resolve(model,name)
            with torch.no_grad(): module.weight.copy_(original.to(module.weight.device))
        gc.collect(); torch.cuda.empty_cache()

def run_experiments():
    if not torch.cuda.is_available(): raise RuntimeError('本番はColab GPUで実行してください')
    if not torch.cuda.is_bf16_supported(): raise RuntimeError('BF16対応GPU（例:L4）を選択してください')
    versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy','datasets']}
    write_json(OUTPUT/'environment.json',dict(versions=versions,python=platform.python_version(),gpu=torch.cuda.get_device_name()))
    for key in MODELS:
        tokens=tokens_load(key); baseline_dir=result_root(key)/'baselines'/key
        pending=[j for j in JOBS if j['model']==key and not completed(run_directory(j))]
        if not pending and completed(baseline_dir):
            print(f'{key}: all saved results reused; no model reload')
            continue
        model=model_load(key)
        try:
            if not completed(baseline_dir):
                loss,count=evaluate(model,tokens)
                baseline_ppl=safe_exp(float(loss.sum()/count.sum()))
                print(f'{key} verified-load baseline PPL: {baseline_ppl:.4f}',flush=True)
                # 緩い異常検知。LRA後の大きいPPLは制限せず、未圧縮baselineだけを確認。
                if not np.isfinite(baseline_ppl) or baseline_ppl>1000:
                    raise RuntimeError('Baseline PPL is abnormal; stopped before LRA. No completed baseline saved.')
                save_eval(baseline_dir,loss,count,dict(model=key,config_hash=CONFIG_HASH,token_sha256=sha(OUTPUT/f'{key}_tokens.npy')))
            with np.load(baseline_dir/'block_losses.npz') as z: baseline=(z['loss_sum'].copy(),z['token_count'].copy())
            for i,job in enumerate(pending):
                print(f"[{i+1}/{len(pending)}] {job['run_id']}",flush=True)
                execute_job(model,key,job,tokens,baseline)
        finally:
            del model; gc.collect(); torch.cuda.empty_cache()
if RUN_EXPERIMENTS: run_experiments()
else: print('本番未実行。計画確認後、RUN_EXPERIMENTS=Trueでこのセルを実行。')

llama: all saved results reused; no model reload


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

[1/55] gemma__E2__all_single__layers.30.mlp.up_proj__step0__1
[2/55] gemma__E2__all_single__layers.30.self_attn.v_proj__step0__0
[3/55] gemma__E2__all_single__layers.30.self_attn.v_proj__step0__1
[4/55] gemma__E2__all_single__layers.31.mlp.down_proj__step0__0
[5/55] gemma__E2__all_single__layers.31.mlp.down_proj__step0__1
[6/55] gemma__E2__all_single__layers.31.mlp.gate_proj__step0__0
[7/55] gemma__E2__all_single__layers.31.mlp.gate_proj__step0__1
[8/55] gemma__E2__all_single__layers.31.self_attn.k_proj__step0__0


: 

## 7. PPLでの対応比較
DE改善率 = 100 × (1 − PPL_DE / PPL_OffDE)。正がDE優位。
信頼区間は同一評価ブロックを4ブロックずつまとめた対応bootstrap（2000回）。
未完了は欠測のまま保持。モデル間のPPL絶対値は比較しません。


In [ ]:
def read_losses(path):
    with np.load(path/'block_losses.npz') as z: return z['loss_sum'].copy(),z['token_count'].copy()

def ppl_bootstrap(a,b):
    al,ac=a; bl,bc=b
    if not np.array_equal(ac,bc): raise ValueError('Token counts differ')
    groups=[slice(i,min(i+4,len(ac))) for i in range(0,len(ac),4)]
    diff=np.array([(bl[g]-al[g]).sum() for g in groups]); counts=np.array([ac[g].sum() for g in groups])
    rng=np.random.default_rng(SEED); values=[]
    for _ in range(BOOTSTRAP_REPEATS):
        idx=rng.integers(0,len(groups),len(groups))
        values.append(100*(-np.expm1(diff[idx].sum()/counts[idx].sum())))
    return np.quantile(values,[.025,.975])

def analyze():
    rows=[]; pairs=[]
    for job in JOBS:
        path=run_directory(job)
        if not completed(path): continue
        r=json.loads((path/'summary.json').read_text())
        r['baseline_ppl']=safe_exp(r['baseline_nll'])
        r['ppl_baseline_ratio']=r['ppl']/r['baseline_ppl'] if r['ppl'] is not None else np.nan
        rows.append(r)
    raw=pd.DataFrame(rows)
    if raw.empty: return raw,pd.DataFrame()
    for key,g in raw.groupby(['model','experiment','variant','scope','step']):
        if len(g)!=2 or set(g.use_de)!={False,True}: continue
        off=g.loc[~g.use_de].iloc[0]; de=g.loc[g.use_de].iloc[0]
        ja=next(j for j in JOBS if j['run_id']==off.run_id); jb=next(j for j in JOBS if j['run_id']==de.run_id)
        if ja['names']!=jb['names'] or off.token_sha256!=de.token_sha256: raise ValueError('Pair mismatch')
        if key[1]=='E2' and (ja['ranks']!=jb['ranks'] or off.params_saved!=de.params_saved): raise ValueError('Ranks mismatch')
        lo,hi=ppl_bootstrap(read_losses(run_directory(ja)),read_losses(run_directory(jb)))
        ratio=safe_exp(de.nll-off.nll)
        pairs.append(dict(zip(['model','experiment','variant','scope','step'],key)) | dict(
            ppl_off=off.ppl,ppl_de=de.ppl,baseline_ppl=off.baseline_ppl,
            ppl_ratio_DE_to_OffDE=ratio,improvement_pct=100*(1-ratio),ci_low_pct=lo,ci_high_pct=hi,
            reduction_off=off.reduction_model,reduction_de=de.reduction_model))
    # 内部損失は各runに保持し、発表用CSVではPPLのみを提示。
    results=raw.drop(columns=['nll','baseline_nll'],errors='ignore'); paired=pd.DataFrame(pairs)
    write_csv(OUTPUT/'results_ppl.csv',results);write_csv(OUTPUT/'paired_ppl.csv',paired)
    print(f'Completed {len(results)}/{len(JOBS)}; pairs={len(paired)}')
    if not paired.empty: display(paired.head(14).round(4))
    return results,paired
RESULTS,PAIRED=analyze()


## 8. PPLの図
単独LRA：ブロックごとのPPL（未圧縮を点線表示）とDE改善率。
累積LRA：横軸は理論削減率、数字は行列数。DE有無でrank・削減率が異なる点に注意。


In [ ]:
def save_figure(fig,name):
    out=OUTPUT/'figures';out.mkdir(exist_ok=True)
    fig.tight_layout()
    for ext in ['png','pdf']: fig.savefig(out/f'{name}.{ext}',dpi=160,bbox_inches='tight')
    plt.show();plt.close(fig)

def plot_results(results,paired):
    if paired.empty: return
    for model,g in paired[paired.experiment=='E2'].groupby('model'):
        g=g.copy();g['block']=g.scope.str.extract(r'layers\.(\d+)\.')[0].astype(int);g['module']=g.scope.str.split('.').str[-1]
        fig,axes=plt.subplots(2,4,figsize=(16,8));axes.flat[-1].axis('off')
        for ax,module in zip(axes.flat,MODULES):
            h=g[g.module==module].sort_values('block')
            ax.plot(h.block,h.ppl_off,'s--',label='OffDE');ax.plot(h.block,h.ppl_de,'o-',label='DE')
            ax.axhline(g.baseline_ppl.iloc[0],color='gray',ls=':',label='Baseline')
            ax.set(title=module,xlabel='Block',ylabel='PPL (log scale)',yscale='log');ax.legend()
        save_figure(fig,f'E2_single_ppl_{model}')
        table=g.pivot(index='block',columns='module',values='improvement_pct').reindex(index=range(len(METRICS[model])//7),columns=MODULES)
        # 極端な悪化で他の差が消えないよう±100%で表示を飽和。正確な値はCSV。
        fig,ax=plt.subplots(figsize=(10,10));cmap=plt.get_cmap('RdBu').copy();cmap.set_bad('lightgray')
        im=ax.imshow(np.ma.masked_invalid(table.values),aspect='auto',cmap=cmap,vmin=-100,vmax=100)
        ax.set_xticks(range(7),MODULES,rotation=30);ax.set_yticks(range(len(table)),table.index)
        ax.set(title=f'{model}: PPL improvement by DE (gray = unmeasured)',ylabel='Block')
        fig.colorbar(im,ax=ax,extend='both',label='100 × (1 − PPL_DE / PPL_OffDE) [%]')
        save_figure(fig,f'E2_improvement_{model}')
    for model,g in results[results.experiment=='E3'].groupby('model'):
        fig,ax=plt.subplots()
        for de,h in g.groupby('use_de'):
            h=h.sort_values('step');ax.plot(100*h.reduction_model,h.ppl,'o-' if de else 's--',label='DE' if de else 'OffDE')
            for row in h.itertuples(): ax.annotate(str(row.step),(100*row.reduction_model,row.ppl),fontsize=8)
        ax.axhline(g.baseline_ppl.iloc[0],color='gray',ls=':',label='Baseline')
        ax.set(xlabel='Theoretical language-model parameter reduction (%)',ylabel='PPL (log scale)',yscale='log',title=f'{model}: descending two-sided bulkKS, DE variance 1');ax.legend()
        save_figure(fig,f'E3_bulk_order_ppl_{model}')
plot_results(RESULTS,PAIRED)


## 9. 保存・終了
結果は別TAGに保存。旧08の結果は変更しません。未完了時は自動切断しません。

In [ ]:
if not PAIRED.empty:
    lines=['# IBIS2026 DE対応比較（自動集計）',f'完了run: {len(RESULTS)}/{len(JOBS)}',
           'PPL改善率は正ならDE優位。CIは評価ブロック群bootstrap。']
    for (model,exp,variant),g in PAIRED.groupby(['model','experiment','variant']):
        lines.append(f'- {model}/{exp}/{variant}: {len(g)} pairs; PPL improved={int((g.improvement_pct>0).sum())}; CI lower>0={int((g.ci_low_pct>0).sum())}')
    lines.append('未完了条件を含む場合は暫定結果。効果方向の件数は独立標本の検定ではない。')
    (OUTPUT/'analysis_summary.md').write_text('\n'.join(lines))
    print('\n'.join(lines))
    if PUBLISH_WANDB:
        import wandb
        if not os.environ.get('WANDB_API_KEY'): os.environ['WANDB_API_KEY']=getpass.getpass('WANDB_API_KEY: ')
        with wandb.init(entity=WANDB_ENTITY,project=WANDB_PROJECT,name=EXPERIMENT_TAG,config={'config_hash':CONFIG_HASH}) as run:
            run.log({'results':wandb.Table(dataframe=RESULTS),'paired':wandb.Table(dataframe=PAIRED)})
            artifact=wandb.Artifact(EXPERIMENT_TAG,type='evaluation')
            for name in ['plan.csv','results_ppl.csv','paired_ppl.csv','manifest.json','analysis_summary.md']:
                if (OUTPUT/name).exists(): artifact.add_file(str(OUTPUT/name))
            run.log_artifact(artifact)
if AUTO_DISCONNECT and RUN_EXPERIMENTS and len(RESULTS)==len(JOBS) and Path('/content').exists():
    from google.colab import runtime
    runtime.unassign()